# Homework Seven (DSC 210, Fall 2025)
Name: Jacob Bauer

Date: 2026-04-26

The goal for this homework is to build a classification model that can identify real and fake news stories. The datasets for this homework are labeled "True.csv" and "Fake.csv" in the Drive folder for homework.

1. Investigate and clean up the datasets. You should consider steps such as:
  - Creating a new column in each dataset indicating if the articles are real or fake
  - Joining the two datasets together reasonably


In [46]:
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns 
import pandas as pd

true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

true_df["true"] = True
fake_df["true"] = False

tf_df = pd.merge(true_df, fake_df, how = "outer")
tf_df.head()

,title,text,subject,date,true
0,\r\r\r\r\r\r\r\nDonald Trump Gets His A** Hand...,"A 10-year-old girl smacked down Donald Trump, ...",News,"February 29, 2016",False
1,\r\r\r\r\r\r\r\nEx-GOP Congressman Shreds Fell...,"At this point, every Republican who refuses to...",News,"June 22, 2017",False
2,\r\r\r\r\r\r\r\nTrump Gets STOMPED For Whining...,Donald Trump has been attacking President Obam...,News,"November 5, 2016",False
3,\r\r\r\r\r\r\r\nWATCH: Democratic Rep Delivers...,Massachusetts Rep. Jim McGovern brought the Ho...,News,"May 4, 2017",False
4,#AfterTrumpImplodes Hashtag Hilariously Imagi...,What will the world be like post-Donald Trump?...,News,"August 5, 2016",False


2. Investigate the article titles. Is there a relationship between words in the title and whether the article is fake or true?

- Process the text in the titles. This should include lowercasing at least, but other steps *may* be appropriate.
- Pick a word that is related to US politics. This word choice is up to you.
- Set up a regular expression that captures the variations of your chosen word.
- Find the proportion of articles that contain variations of your chosen word, for real and fake articles separately.
- Do these proportions indicate any immediate difference between real and fake articles, in terms of their titles?

In [47]:
tf_df["title"] = tf_df["title"].str.lower()
tf_df["text"] = tf_df["text"].str.lower()

pattern = r"\bpresiden\w*"
tf_df["has_president"] = tf_df["text"].str.contains(pattern, case=False, na=False)

proportions_tf = tf_df.groupby("true")["has_president"].mean()
print(proportions_tf)

# Fake news articles mention president 53% of the time
# True news articles mention president 67% of the time
# This indicated that true news articles tend to mention the word president more often than fake news articles

true
False    0.534262
True     0.671242
Name: has_president, dtype: float64


3. Build classification model to predict article authenticity (real or fake).

Some notes:
- Follow the workflow used in class.
- Make reasonable decisions about hyperparameters, and **justify/explain your decisions**. You might try out a few different hyperparameters to judge performance (hyperparameter tuning).
- Comment on the model performance. Relate the performance to a naive model.

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction import text
import re

def clean_source_metadata(text):
    # Removes "WASHINGTON (Reuters) -" or similar patterns at the start
    text = re.sub(r'^.*?\(Reuters\)\s?-', '', text)
    # Removes common website artifacts like "Featured Image via..."
    text = re.sub(r'Featured Image via.*', '', text)
    return text


tf_df['text'] = tf_df['text'].apply(clean_source_metadata)

cheating_words = [
    'reuters', 'washington', 'said', 'video', 'image', 
    'featured', 'edt', 'wednesday', 'nov', 'gop', 'hillary'
]

stop_words = list(text.ENGLISH_STOP_WORDS.union(cheating_words))

# Preprocessing
X = tf_df['title'].fillna('')
y = tf_df['true']

# Test Train sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Vectorize 
tfidf = TfidfVectorizer(
    stop_words=stop_words, # Eliminates words with high coefficients
    max_features=5000, # Limits model to 5000 words 
    ngram_range=(1, 2) # Allows the model to look at both one word pairs and two word pairs 
)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# Logistic Regression
for c_val in [0.1, 1.0, 10.0]: # Gives the different strength levels of model where 0.1 is basic and 10 is complex.
    lr = LogisticRegression(C=c_val, max_iter=1000) # Searches for best results after running the model 1000 times
    lr.fit(X_train_vec, y_train)
    print(f"Accuracy with C={c_val}: {lr.score(X_test_vec, y_test):.4f}")

final_model = LogisticRegression(C=10.0, max_iter=1000)
final_model.fit(X_train_vec, y_train)

y_pred = final_model.predict(X_test_vec)
print(f"Final Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# Get the feature names from the vectorizer
feature_names = tfidf.get_feature_names_out()

# Get the coefficients from the final model
coefs = final_model.coef_[0]

# Create a DataFrame to view the top words
coef_df = pd.DataFrame({'word': feature_names, 'coefficient': coefs})

print("Top words for 'True' class:")
print(coef_df.sort_values(by='coefficient', ascending=False).head(10))

print("\nTop words for 'False' class:")
print(coef_df.sort_values(by='coefficient', ascending=True).head(10))

# Model performance was 93% which compared to the naive model is much better. The Naive model based on the previous code
# states that the performce of guessing true news with the word president would be 67%.

Accuracy with C=0.1: 0.9130
Accuracy with C=1.0: 0.9337
Accuracy with C=10.0: 0.9379
Final Model Accuracy: 0.9379
              precision    recall  f1-score   support

       False       0.94      0.94      0.94      5929
        True       0.94      0.93      0.93      5296

    accuracy                           0.94     11225
   macro avg       0.94      0.94      0.94     11225
weighted avg       0.94      0.94      0.94     11225

Top words for 'True' class:
               word  coefficient
1530        factbox    11.077875
3780           says     8.603375
2219  islamic state     8.328500
2852        myanmar     8.310384
4715          urges     8.127187
3686       rohingya     7.721620
4648         turkey     7.244568
3094       pakistan     7.221639
586         britain     7.128065
2249          japan     6.990557

Top words for 'False' class:
           word  coefficient
561    breaking   -14.192430
2328       just   -12.653431
4830      watch   -11.850226
3443     racist    -9.